# Justice.gov.et PDF Downloader (Google Colab)

Run this notebook to download regulation PDFs when your home/server IP is blocked.

**Steps:** Runtime → Run all (or run each cell top to bottom).

## 1. Test network (must return HTTP 200)

In [ ]:
!curl -L --max-time 30 --http1.1 \
  -A "Mozilla/5.0 Chrome/126.0.0.0 Safari/537.36" \
  "https://justice.gov.et/en/laws/regulations/" \
  -o /tmp/test.html -w "\nHTTP: %{http_code}\n"

## 2. Clone repo & install deps

In [ ]:
!git clone https://github.com/blengebre-ug-hub/pdf_reader.git
%cd pdf_reader/justice_regulation_downloader
!pip install -q pypdf
!apt-get -qq install -y curl > /dev/null

## 3. (Optional) Mount Google Drive to save PDFs permanently

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = "/content/drive/MyDrive/justice_pdfs"
os.makedirs(SAVE_DIR, exist_ok=True)
print("PDFs will save to:", SAVE_DIR)

## 4. Run downloader

Choose **one**:
- **Fresh crawl** from regulations page (slow, discovers URLs)
- **`--download-only`** if you uploaded `download_queue.sqlite3` to Drive

In [ ]:
# Change OUTPUT if you mounted Drive (step 3)
OUTPUT = SAVE_DIR if 'SAVE_DIR' in dir() else "pdfs"

!python3 downloader.py \
  --base-url "https://justice.gov.et/en/laws/regulations/" \
  --output-dir {OUTPUT} \
  --workers 20

## 5. Check progress

In [ ]:
!sqlite3 {OUTPUT}/download_queue.sqlite3 \
  "SELECT kind, status, COUNT(*) FROM discovered_urls GROUP BY kind, status;"

import glob
pdfs = glob.glob(f"{OUTPUT}/*.pdf")
print(f"PDF files on disk: {len(pdfs)}")

## 6. Download zip to your PC (if not using Drive)

In [ ]:
!zip -r /content/justice_pdfs.zip {OUTPUT}/*.pdf 2>/dev/null | tail -1
from google.colab import files
files.download('/content/justice_pdfs.zip')